In [11]:
# Load libraries and configs
library(tidyverse)
library(readxl)
readRenviron("../../.Renviron")
source(file.path("../../config.R"))

cat("This notebook last updated at:", format(Sys.time()))

This notebook last updated at: 2026-07-21 19:34:25

# JRN Plant List - Notes and validation

This document has notes and validation checks for the new JRN plant list located at `jornada_im/dataprep/jrn520_taxa/plants/jrn_plant_list_MAIN.xlsx`. This version of the JRN plant list was derived from joining two source tables:

1. The most recent **EDI plant list**, (`JRN vascular plant species list.csv` or `jrn_plant_list_MAIN_YYYYMMDD.csv` in later versions) which itself came from a reformatted version of John Anderson’s list (Plntalfa_table_edit.xlsx) and Justin VanZee’s list (plantlistJER), and was maintained in Excel for a few years. This is a more inclusive list that includes over 500 taxa of the region. 
2. **John's plant list** that is maintained in the Field Crew Sharepoint directory (filename = `0_test-merge1.xlsx`). This is a less-inclusive list, focusing on taxa observed by the JRN field crew. It was probably also derived at some point from the "Plntalfa" list.

These two files were merged in R (see `reformat_main_plants.R` for details), and plant codes (LTER and USDA) and scientific names were then validated against the USDA Plants database (March 2026 version) and Kelly Allred's Flora of the Jornada Plain (9th edition) to produce a table of 573 taxa in the JRN plant list (`jrn_plants` in the spreadsheet) Plant traits mostly rely on **John's plant list**, but newer trait columns from the **EDI plant list** have been retained. Various synonyms have been extracted from both source lists and are collected in a new synonyms table that includes both out-of-date taxonomy, and accepted alternative taxonomies from USDA Plants or Allred when this list deviates from those for some reason. 

## Table descriptions

There are two related tables in the plant list. The JRN plant list (`plant_list` in the spreadsheet) and the list of synonyms (`plant_synonyms`). Descriptions for the two tables and their columns are below.

### The JRN plant list (`plant_list` sheet)

This is the main JRN plant list. It will be published as the next version of the **EDI plant list** when we're ready. There are 573 taxa, one row per taxon, sorted by `sciname`.

| Column | Derivation |
|--------|------------|
| `family` | From **John's plant list**, falling back to **EDI plant list** |
| `sciname` | Binomial name + infraspecific rank stripped of authority, derived by regex from `sciname_auth` |
| `sciname_auth` | Binomial name + infraspecific rank, with taxonomic authority. From **EDI plant list**, falling back to **John's plant list**, then USDA Plants; replaced with `sciname_auth_usda` (and old value moved to `alias`) where `sciname_usda_match` is FALSE, `sciname_auth_follows == "Other"`, and a USDA name is available |
| `usda_code` | From **John's plant list** or **EDI plant list** depending on manual review (see `use_john`/`use_main` vectors); resolved to accepted USDA symbol in most cases |
| `lter_code` | 4-character Jornada LTER species code; primary key; from **EDI plant list** (full join with **John's plant list**) |
| `common_name` | Common names from **John's plant list**, filled from USDA Plants then **Allred plant list**; new names appended as semicolon-delimited list, case-insensitive deduplication applied |
| `habit` | Categorical (`A`=annual, `B`=biennial, `P`=perennial); from **John's plant list**, falling back to **EDI plant list** |
| `form` | Categorical (`FERN`, `FORB`, `GRASS`, `LF-SU`, `S-SHR`, `SHRUB`, `ST-SU`, `TREE`, `VINE`); from **John's plant list**, falling back to **EDI plant list** |
| `cpath` | Categorical — photosynthetic pathway (`C3`, `C4`, `CAM`, `PAR`); from **John's plant list**, falling back to **EDI plant list** |
| `nativity` | Categorical (`native`, `introduced`); from **EDI plant list** |
| `habitat` | From **EDI plant list** |
| `phenology` | From **EDI plant list** |
| `reproduction` | Categorical (`seed`, `spore`); from **EDI plant list** |
| `lter_observed` | Categorical (`present`, `not observed`); from **EDI plant list** |
| `sciname_auth_follows` | Categorical — which source the accepted `sciname_auth` agrees with (`USDA Plants`, `Allred`, `Other`) |
| `usda_code_is_syn` | Boolean; `TRUE` if `usda_code` is a USDA synonym symbol rather than an accepted symbol |
| `note` | Free-text notes from **EDI plant list**, augmented during manual editing |

The last three columns are useful for diagnostic purposes. `usda_code_is_syn` indicates which of the codes in `usda_code` is a synonym in the USDA Plants database. Taxonomy for these species has probably been updated and there is a new accepted code for the taxa. This will be listed in the `synonyms` table. `sciname_auth_follows` indicates when the `sciname_auth` column, ultimately derived from the LTER taxonomy in the previous **EDI plant list**, matches the USDA Plants taxonomy associated with that USDA code, Kelly Allred's taxonomy, or something else. Where "Other", the taxon name may be following a preferred local authority, outdated taxonomy (old USDA taxa), or there may be a subtle difference between `sciname_auth` and the same taxa in USDA or Allred taxonomy (such as author abbreviations or infrapecific rank delimiter). Also note that most of the time, USDA plants and Allred agree, they just write scientific names and authorities slightly differently. The `notes` column collects notes about conflicts between taxonomic authorities, duplicate codes, and other issues.

---

### Synonyms (`plant_synonyms` sheet)

This table contains collected synonyms for the taxa in the JRN plant list and will be published alongside the JRN plant list. The values in the `usda_code` and `lter_code` columns are drawn from the main plant list (foreign keys to the same columns in `plant_list`). There is one row per synonym, and there may be 0-to-many synonyms per taxon in the JRN plant list. The table is de-duplicated but there are still some near-duplicate synonyms that can be removed after manual review. Sorted by `lter_code`.

| Column | Derivation |
|--------|------------|
| `usda_code` | Accepted USDA code for the taxon (from `plant_list`) |
| `lter_code` | LTER code for the taxon (from `plant_list`) |
| `sciname_auth` | The synonym scientific name (one alias entry per row, split from the semicolon-delimited `alias` column; or an old `Direct USDA Code` entry from **John's plant list**) |
| `usda_code_syn` | USDA symbol corresponding to the synonym name: matched by sciname against USDA synonym rows; when `usda_code_is_syn` is TRUE, falls back to matching against USDA accepted rows and taking `usda_code`; `NA` if no match found |


In [12]:
# Path to the plant list files (jrn520)
plants_path <- file.path(im_path, "dataprep", "jrn520_taxa", "plants")
# Load the plant list and synonyms
mainpl <- read_excel(file.path(plants_path, "jrn_plant_list_MAIN.xlsx"), sheet='plant_list',
                        skip=4, na = c(".", "NA"))
syn <- read_excel(file.path(plants_path, "jrn_plant_list_MAIN.xlsx"), sheet='plant_synonyms',
                        skip=4, na = c(".", "NA"))

## LTER and USDA code validation checks

Ideally, all taxa included in the main JRN plant list should have one unique LTER code and a unique USDA Plants code. This isn't always the case:

* The same LTER code may be used to refer to two unique taxa (duplicate LTER codes)
* Multiple LTER codes may refer to the same taxon (synonym LTER codes, duplicate USDA codes)
* A USDA code may refer to the same taxon twice, but multiple LTER codes indicate the LTER codes are synonyms (converse of the above).
* Some taxa are missing an LTER code.

Summary data for these cases are below, and cases are examined in depth after.

In [13]:
# Check: Are there duplicate lter codes?
lter_code_dup <- mainpl |>
  filter(!is.na(lter_code)) |>
  group_by(lter_code) |> filter(n() > 1) |>
  ungroup() |> arrange(lter_code) |>
  select(lter_code, usda_code, sciname_auth)
cat("Number of duplicate lter_code values in mainpl:", nrow(lter_code_dup)/2, "\n")


# Check: Duplicate LTER codes with multiple USDA codes
lter_multi_usda <- mainpl |>
  filter(!is.na(lter_code), !is.na(usda_code)) |>
  group_by(lter_code) |> filter(n_distinct(usda_code) > 1) |>
  ungroup() |> arrange(lter_code) |>
  select(lter_code, usda_code, sciname_auth)
cat("Number of duplicate lter_code values with multiple usda_codes:", n_distinct(lter_multi_usda$lter_code), "\n")
#if (nrow(lter_multi_usda) > 0) print(lter_multi_usda, n=50)

# Check: Are there duplicate usda codes?
usda_code_dup <- mainpl |>
  filter(!is.na(usda_code)) |>
  group_by(usda_code) |> filter(n() > 1) |>
  ungroup() |> arrange(usda_code) |>
  select(usda_code, lter_code, sciname_auth)
cat("Number of duplicate usda_code values in mainpl:", nrow(usda_code_dup)/2, "\n")

# Check: Duplicate USDA codes with multiple LTER codes
usda_multi_lter <- mainpl |>
  filter(!is.na(usda_code), !is.na(lter_code)) |>
  group_by(usda_code) |> filter(n_distinct(lter_code) > 1) |>
  ungroup() |> arrange(usda_code) |>
  select(usda_code, lter_code, sciname_auth)
cat("Number of duplicate usda_code values with multiple lter_codes:", n_distinct(usda_multi_lter$usda_code), "\n")

# Check: Missing (NA) LTER codes
lter_na <- mainpl |>
  filter(is.na(lter_code)) |>
  ungroup() |> arrange(usda_code) |>
  select(lter_code, usda_code, sciname_auth)
cat("Number of missing (NA) lter_codes:", nrow(lter_na), "\n")

# Check: Missing USDA codes
usda_na <- mainpl |>
  filter(is.na(usda_code)) |>
  ungroup() |> arrange(usda_code) |>
  select(usda_code, lter_code, sciname_auth)
cat("Number of missing usda_codes:", nrow(usda_na), "\n")

# Check: Synonym (outdated) USDA codes
usda_syn <- mainpl |>
  filter(usda_code_is_syn) |>
  ungroup() |> arrange(usda_code) |>
  select(usda_code, lter_code, sciname_auth)
cat("Number of synonym (outdated) usda_codes:", nrow(usda_syn), "\n")

Number of duplicate lter_code values in mainpl: 0 
Number of duplicate lter_code values with multiple usda_codes: 0 
Number of duplicate usda_code values in mainpl: 1 
Number of duplicate usda_code values with multiple lter_codes: 0 
Number of missing (NA) lter_codes: 11 
Number of missing usda_codes: 0 
Number of synonym (outdated) usda_codes: 29 


### Duplicate LTER codes

Below is a table of all duplicate LTER codes in the list. In these cases, the same LTER code appears multiple times in the plant list. **This may lead to many-to-one relationships when joining to other data!**

In [14]:
if (nrow(lter_code_dup) > 0) print(lter_code_dup, n=50)

All of these are LTER codes that refer to multiple taxonomic entities, either different species or infraspecific taxa. Duplicate codes that have been "resolved" as of 2026-07-21:  

`ACCO`, `BOCC`, `BOCU`, `CHLI`, `ERCU`, `PRGL`, `XAST` -  which referred to two infraspecific ranks of the same species, and all but one (`BOCC`) had unique USDA codes for the infraspecific taxa. The `BOCC` infraspecifics were not recognized by USDA Plants. One duplicate code for all these taxa have now been assigned an "NA" LTER code if they are thought to be unobserved or rare at the Jronada. If they become officially recognized and observed taxa on the Jornada, we will want to assign a unique LTER code for each taxon.

For `TECO` and `MESC`, the same code referred to a species and an infraspecific rank of that species. These duplicate LTER codes were more a product of discrepancies between the taxonomic authorities that were used to compose the list (John's list vs Justin's vs Darren's). In this case, the infraspecific taxa are not recognized with an LTER code and assigned "NA" for consistency's sake. If they are observed at some point this will change.

In the case of `ECTR` and `SILE`, taxa have been split or are disputed between different authorities. For `ECTR`, prior authorities have recognized _Echinocereus triglochidiatus var. gurneyi_, but Allred disputes this and only recognizes _Echinocereus coccineus var. coccineus_. For `SILE`, the taxa once known as _Sida lepidota_ has been split into two species (`MALE2` and `MASA3`) in USDA Plants and Allred. The unobserved or less common taxon has been given an "NA" LTER code.

### Missing LTER codes

Which were formerly the duplicates above.

In [15]:
if (nrow(lter_na) > 0) print(lter_na, n=50)

# A tibble: 11 × 3
   lter_code usda_code sciname_auth                                             
   <chr>     <chr>     <chr>                                                    
 1 NA        ACCOP9    Acacia constricta var. paucispina Woot. & Standl.        
 2 NA        BOCO2     Boerhavia coulteri (Hooker f.) S. Watson var. palmeri (S…
 3 NA        BOCUC     Bouteloua curtipendula (Michx.) Torr. var. caespitosa Go…
 4 NA        CHLIA     Chilopsis linearis (Cav.) Sweet. subsp. arcuata Fosberg  
 5 NA        ECTRG2    Echinocereus triglochidiatus Engelm. var. gurneyi L.D. B…
 6 NA        ERCUC2    Eragrostis curvula (Schrad.) Nees var. conferta Stapf.   
 7 NA        MASA3     Malvella sagittifolia (Gray) Fryxell                     
 8 NA        MESCL     Menodora scabra Gray var. laevis (Woot. & Standl.) Steye…
 9 NA        PRGLG     Prosopis glandulosa Torr. var. glandulosa                
10 NA        TECOA     Tetraclea coulteri Gray var. angustifolia (Woot. & Stand…
11 NA    

### Duplicate USDA codes

Below is a table of duplicate USDA codes. In these cases, the same USDA code appears multiple times in the plant list. **This may lead to many-to-one relationships when joining to other data!** This can also help identify whether duplicate LTER codes, such as shown above, are synonyms for the same taxa.

In [16]:
if (nrow(usda_code_dup) > 0) print(usda_code_dup)

# A tibble: 2 × 3
  usda_code lter_code sciname_auth                                              
  <chr>     <chr>     <chr>                                                     
1 BOCO2     BOCC      Boerhavia coulteri (Hooker f.) S. Watson var. coulteri    
2 BOCO2     NA        Boerhavia coulteri (Hooker f.) S. Watson var. palmeri (S.…


Many other duplicates have been resolved by removing deprecated LTER codes that referred to an outdated synonym for an accepted USDA Plants taxon, and by adding correct (though outdated) USDA synonym code to taxa with a taxonomy change not reflected in the JRN plant list yet. These taxa may need to be updated and the outdated LTER codes retired. One remaining duplicate USDA code (`BOCO2`) is the closest USDA code match referring to two infraspecific taxa of the same species, both of which share the same LTER code (`BOCC`). Here it might make sense to choose a new LTER code for one of the taxa if both are important.

### Synonym USDA codes

As noted above, some entries of the JRN plant list refer to synonym USDA codes, which may be outdated or in dispute with other taxonomic authorities. A list is below - each of the USDA codes shown is a synonym.

In [17]:
if (nrow(usda_syn) > 0) print(usda_syn, n=30)

# A tibble: 29 × 3
   usda_code lter_code sciname_auth                                             
   <chr>     <chr>     <chr>                                                    
 1 ACCO2     ACCO      Acacia constricta Benth.                                 
 2 ACCOP9    NA        Acacia constricta var. paucispina Woot. & Standl.        
 3 ACNE4     ACNE      Acacia neovernicosa Isely                                
 4 ARPAD     ARDI      Aristida pansa Woot. & Standl. forma dissita (I.M.Johnst…
 5 ASALP     ASWO      Astragalus allochrous Gray var. playanus (M.E. Jones) Is…
 6 BOTO2     BOTR      Boerhavia torreyana (S. Watson) Standl.                  
 7 COSC      COSA      Commicarpus scandens (L.) Standl.                        
 8 ECTRG2    NA        Echinocereus triglochidiatus Engelm. var. gurneyi L.D. B…
 9 ELSM3     ELSM      Elymus smithii (Rydb.) Gould                             
10 EQHYA2    EQHY      Equisetum hyemale L. subsp. affine (Engelm.) Calder & Ro…
11 ERCUC2

## Scientific name and authority validation checks

Now lets look at whether the scientific name and authority (`sciname_auth` column) for each taxon matches the taxonomy in USDA Plants or not.

In [18]:
# Load USDA plants list and create lookup tables
usda <- read_csv(file.path(plants_path, 'usda_plantlst_20260318.txt')) |>
  rename_with(tolower) |>
  rename(usda_code = symbol, usda_code_syn = `synonym symbol`,
         sciname_auth = `scientific name with author`) |>
  mutate(sciname_auth = str_replace(sciname_auth, " ssp. ", " subsp. "))

# Helper to strip authority from sciname_auth — reused in both lookups below
derive_sciname <- function(x) {
  base  <- str_extract(x, "^[A-Z][a-z-]+ [a-z×-]+")
  infra <- str_extract(x, "(?<= )(?:subsp\\.|ssp\\.|var\\.|f\\.|forma) [a-z-]+")
  if_else(is.na(infra), base, paste(base, infra))
}

# Lookup 1: USDA synonym rows — sciname -> usda_code_syn
usda_syn_lookup <- usda |>
  filter(!is.na(usda_code_syn)) |>
  mutate(sciname_syn = derive_sciname(sciname_auth)) |>
  select(usda_code_syn, sciname_syn, sciname_auth)

# Lookup 2: USDA accepted rows — sciname -> usda_code (used when usda_code_is_syn is TRUE)
usda_accepted_lookup <- usda |>
  filter(is.na(usda_code_syn)) |>
  mutate(sciname_syn = derive_sciname(sciname_auth)) |>
  select(usda_code_accepted = usda_code, sciname_syn, sciname_auth)

Rows: 93157 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (5): Symbol, Synonym Symbol, Scientific Name with Author, Common Name, F...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [19]:
# Check: exact sciname_auth match against USDA Plants
# Accepted taxa: compare against accepted sciname_auth via usda_accepted_lookup
# Synonym taxa: compare against synonym row sciname_auth via usda_syn_lookup
sciname_auth_check <- bind_rows(
  mainpl |>
    filter(!usda_code_is_syn) |>
    left_join(usda_accepted_lookup |> select(usda_code_accepted, sciname_auth_usda = sciname_auth),
              by = c("usda_code" = "usda_code_accepted")),
  mainpl |>
    filter(usda_code_is_syn) |>
    left_join(usda_syn_lookup |> select(usda_code_syn, sciname_auth_usda = sciname_auth),
              by = c("usda_code" = "usda_code_syn"))
) |>
  mutate(sciname_auth_usda_match = sciname_auth == sciname_auth_usda)

cat("Number of taxa where sciname_auth matches USDA Plants exactly:", sum(sciname_auth_check$sciname_auth_usda_match, na.rm = TRUE), "\n")
cat("Number of taxa where sciname_auth does not match USDA Plants:", sum(!sciname_auth_check$sciname_auth_usda_match, na.rm = TRUE), "\n")
cat("Number of taxa where sciname_auth could not be checked (no USDA entry found):", sum(is.na(sciname_auth_check$sciname_auth_usda_match)), "\n")

Number of taxa where sciname_auth matches USDA Plants exactly: 400 
Number of taxa where sciname_auth does not match USDA Plants: 166 
Number of taxa where sciname_auth could not be checked (no USDA entry found): 0 


In [20]:
# For taxa that could not be checked, they may not be listed as a synonym in mainpl
sciname_auth_check[is.na(sciname_auth_check$sciname_auth_usda_match),]


family,sciname,sciname_auth,usda_code,lter_code,common_name,habit,form,cpath,nativity,habitat,phenology,reproduction,lter_observed,sciname_auth_follows,usda_code_is_syn,note,sciname_auth_usda,sciname_auth_usda_match
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<lgl>,<chr>,<chr>,<lgl>


## External list validation

There are some other lists to validate against, both historical lists and currently used ones:

* The **Plntalfa list** (`Plntalfa_20220303.txt`) is the most recent update to the Jornada plant list that is provided on the LTER website and has been historically used for many years by the field crew. Note that we are using a version extracted from the text file by AI (`Plntalfa_extract_checklist_gm.xlsx`), but it has been human-checked.
* The **NPP list** comes from a file in the data processing pipeline for NPP data (project jrn011; `function_list.csv`) that was maintained by Deb Peters group.
* The **Field codes list** is a dataset (`lter_field_codes_MAIN.xlsx`) maintained by the JRN IM team to document codes used in raw datasets and map them to accepted taxonomic codes.

In [21]:
# The Plntalfa list (extracted from John's 2022 version)
plntalfa <- read_excel(file.path(plants_path, "archive",
                            "Plntalfa_extract_checklist_gm.xlsx"), na = c(".", "NA", ""))

# The JRN LTER field codes list
fieldlist <- read_excel(file.path(im_path, "dataprep", "jrn520_taxa", "fieldcodes",
                            "lter_field_codes_MAIN.xlsx"), skip=2)

# The plant list/trait table from the NPP project
jrn011list <- read.csv(file.path(im_path, "dataprep", "jrn011_npp", "anpp", "function_list.csv"), 2,
                       stringsAsFactors = FALSE, na = c(".", "NA", "")) |>
  rename(cpath=path, lter_code=spp)

### Comparison to the Plntalfa list

What plant codes in the **Plntalfa list** are absent from the current JRN plant list?

In [22]:
# What LTER codes from plntalfa are not in the JRN plant list?
plntalfa[!plntalfa$lter_code %in% mainpl$lter_code,c("sciname_auth", "lter_code")]

sciname_auth,lter_code
<chr>,<chr>
None - plants absent from segment,NONE
Rhamnus betulaefolia Greene,RHBE
Road,ROAD
Soil pit (disturbed soil),SOIL
Verbena ambrosifolia Rydb.,VEAM


* _Rhamnus betulaefolia_ (RHBE code, and now Frangula betulifolia (Greene)) appears in Plntalfa, but I have not found the same taxon in any other lists, including **John's list** or the **EDI list**. It also does not appear in the Allred Jornada Plain flora (4th ed). Presumably it was removed at some point and no entry was made in the changelog.
* _Verbena ambrosifolia_ (VEAM) is an outdated synonym for _Verbena wrightii_ (VEWR; which is now _Glandularia bipinattifida_, GLBIC). There is an entry in the changelog that the code was removed and the taxa merged with VEWR in 2018, but the entry remains in the Plntalfa list. This is probably just as an oversight and it is removed in the JRN plant list.

**Now evaluate the `lter_observed` column.**

The JRN plant list has an `lter_observed` column that was derived from the **EDI list** and indicates what taxa have been observed at the Jornada by the LTER field crew. Plants listed as "present" in this column should mirror the taxa listed in the **Plntalfa** list. Lets check how equivalent they actually are.

In [23]:
# Are Plntalfa and the main list where taxa are present roughly equivalent?
present <- mainpl |> filter(lter_observed=="present")
cat("Number of taxa in the plant list where lter_code == present:", nrow(present), "\n")
cat("From these present taxa, these are not found in Plntalfa: \n")
present[!present$lter_code %in% plntalfa$lter_code,c("sciname_auth", "lter_code")]

Number of taxa in the plant list where lter_code == present: 357 
From these present taxa, these are not found in Plntalfa: 


sciname_auth,lter_code
<chr>,<chr>
Eragrostis curvula (Schrad.) Nees var. curvula,ERCU
Pectis cylindrica (Fernald) Rydb.,PECY


In [24]:
cat("and these codes from Plntalfa are not in the present taxa: \n")
plntalfa[!plntalfa$lter_code %in% present$lter_code,c("sciname_auth", "lter_code")]

and these codes from Plntalfa are not in the present taxa: 


sciname_auth,lter_code
<chr>,<chr>
Early data coded as CAHA,CAHA
None - plants absent from segment,NONE
Rhamnus betulaefolia Greene,RHBE
Road,ROAD
Soil pit (disturbed soil),SOIL
Verbena ambrosifolia Rydb.,VEAM


Pectis cylindrica (PECY) was added to plant list in 2022-09-12, after the last edition of **Plntalfa**, so this makes sense. The missing codes from **Plntalfa** have all been removed or deprecated intentionally in the JRN plant list, except for RHBE and CAHA. RHBE is an unknown discrepancy, while CAHA is a lingering deprecated code, an early synonym for _Callandria humilis_, in **Plntalfa**. Note that there is a CAHA in the JRN plant list (_Calylophus hartwegii_), but it is listed as "not observed".

### Comparison to the Field codes list

We want to make sure the **Field codes list** has all LTER codes, whether in use or no longer in use. Is anything from the current JRN plant list missing there?

In [25]:
# Are any lter codes in the main list not covered in the field code list?
mainpl[!mainpl$lter_code %in% fieldlist$field_code, c("sciname_auth", "lter_code")]

sciname_auth,lter_code
<chr>,<chr>
Acacia constricta var. paucispina Woot. & Standl.,NA
Aloina rigida (Hedw.) Limpr.,ALRI
Boerhavia coulteri (Hooker f.) S. Watson var. palmeri (S. Watson) Spellenberg,NA
Bouteloua curtipendula (Michx.) Torr. var. caespitosa Gould & Kapadia,NA
Bryum argenteum Hedw.,BYAR
Cheilanthes lindheimeri Hook.,CHLN
Chilopsis linearis (Cav.) Sweet. subsp. arcuata Fosberg,NA
Desmatodon convolutus (Brid.) Grout,DECN
Desmatodon guepinii Bruch & Schimp.,DEGU


All of these codes are spore plants that are fairly new additions to the plant list and need to be added to the **Field codes list**. We probably should wait until all have official LTER codes.

### Compare plant traits

There are several columns of plant traits in the JRN plant list, including `form`, `habit`, and `cpath`. These are also found in **Plntalfa** and the **NPP list**. We want to make sure the current plant list has the most up-to-date codes and can be used as reference source for these traits. Below we do a merge between the plant list and the other two lists and show the discrepancies.

In [26]:
# ── Merge main JRN plant list and NPP on lter_code ────────────────────────────────────────────────────────
# Inner join keeps only codes present in both lists.
# Use full_join() if you also want codes that exist in only one list.
mergedNPP <- inner_join(
  mainpl  |> select(lter_code, habit, form, cpath),
  jrn011list |> select(lter_code, habit, form, cpath),
  by     = "lter_code",
  suffix = c("_main", "_npp")
)

# ── Flag discrepancies ────────────────────────────────────────────────────────
discrepanciesNPP <- mergedNPP |>
  mutate(
    diff_habit = habit_main != habit_npp | xor(is.na(habit_main), is.na(habit_npp)),
    diff_form  = form_main  != form_npp  | xor(is.na(form_main),  is.na(form_npp)),
    diff_cpath = cpath_main != cpath_npp | xor(is.na(cpath_main), is.na(cpath_npp))
  ) |>
  filter(diff_habit | diff_form | diff_cpath)

cat("Look at discrepancies between the current plant list (_main) and the NPP list (_NPP)\n")
discrepanciesNPP

Look at discrepancies between the current plant list (_main) and the NPP list (_NPP)


lter_code,habit_main,form_main,cpath_main,habit_npp,form_npp,cpath_npp,diff_habit,diff_form,diff_cpath
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<lgl>,<lgl>,<lgl>
ARFE,P,GRASS,C4,P,GRASS,NA,FALSE,FALSE,TRUE
BOBR,P,GRASS,C4,P,GRASS,NA,FALSE,FALSE,TRUE
KOSP,P,SHRUB,C3,P,SHRUB,NA,FALSE,FALSE,TRUE
LIAU,A,FORB,NA,A,FORM,NA,FALSE,TRUE,NA
CAHA,NA,NA,NA,P,S-SHR,C3,TRUE,TRUE,TRUE


In [27]:
# ── Merge main JRN plant list and Plntalfa on lter_code ────────────────────────────────────────────────────────
# Inner join keeps only codes present in both lists.
# Use full_join() if you also want codes that exist in only one list.
mergedPa <- inner_join(
  mainpl  |> select(lter_code, habit, form, cpath),
  plntalfa |> select(lter_code, habit, form, cpath),
  by     = "lter_code",
  suffix = c("_main", "_Pa")
)

# ── Flag discrepancies ────────────────────────────────────────────────────────
discrepanciesPa <- mergedPa |>
  mutate(
    diff_habit = habit_main != habit_Pa | xor(is.na(habit_main), is.na(habit_Pa)),
    diff_form  = form_main  != form_Pa  | xor(is.na(form_main),  is.na(form_Pa)),
    diff_cpath = cpath_main != cpath_Pa | xor(is.na(cpath_main), is.na(cpath_Pa))
  ) |>
  filter(diff_habit | diff_form | diff_cpath)

cat("Look at discrepancies between the current plant list (_main) and Plantalfa (_Pa)\n")
discrepanciesPa

Look at discrepancies between the current plant list (_main) and Plantalfa (_Pa)


lter_code,habit_main,form_main,cpath_main,habit_Pa,form_Pa,cpath_Pa,diff_habit,diff_form,diff_cpath
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<lgl>,<lgl>,<lgl>
ARLO,P,GRASS,C4,NA,NA,NA,TRUE,TRUE,TRUE
ARNE,P,GRASS,C4,NA,NA,NA,TRUE,TRUE,TRUE
ARWR,P,GRASS,C4,NA,NA,NA,TRUE,TRUE,TRUE
ARHA,P,GRASS,C4,NA,NA,NA,TRUE,TRUE,TRUE
BOBR,P,GRASS,C4,P,GRASS,NA,FALSE,FALSE,TRUE
BOTO,P,GRASS,C4,NA,NA,NA,TRUE,TRUE,TRUE
DAPU,P,GRASS,C4,NA,NA,NA,TRUE,TRUE,TRUE
ERMI,A,GRASS,C4,NA,NA,NA,TRUE,TRUE,TRUE
ERLA,P,SHRUB,C3,NA,NA,NA,TRUE,TRUE,TRUE


**No major causes for concern here.** Both earlier lists seem to have mostly the same plant traits as the current JRN plant list. Where there are discrepances it seems to be because the earlier lists lack traits that have been added to later editions, or the discrepancy is with deprecated codes (CAHA is an outdated code in the Plntalfa list).

## Open questions - these are slightly out of date or resolved...

1. Missing LTER codes - 5 for spore plants with conflicting characters
    - One, _Astrolepis sinuata var sinuata_ is already has its parent species in the list with LTER code NOSI. Plntalfa and Allred don't list it. Potentially we could drop the variety and retain species rank only (as NOSI). _2026-06-12_ ...RESOLVED 2026-07-21
2. Duplicate LTER codes: 11 ...RESOLVED 2026-07-21
    - 7 are for two infraspecific names for the same species (ACCO, BOCC, BOCU, CHLI, ERCU, PRGL, XAST). One of these (BOCC) has infraspecifics recognized in Allred but not by USDA Plants (meaning the USDA codes are the same).
    - 2 are for a species and an infraspecific of that species (MESC, TECO)
    - 1 is for two species, with one species disputed (ECTR, see note below)
    - 1 is for two species in which the taxon split (SILE)
    - John and Conrad would like to look at these and make determinations on what to list as "present" at the Jornada site. For example, `PRGL` usually is _var. torreyana_ but they are unsure if _var. glandulosa_ may be present in some areas. Infraspecifics that are not observed should be listed as so and perhaps their LTER code should be changed to NA. When both are observed, it may still be ok to assign one (the less common one, perhaps) an NA code, or we could list the parent species with the LTER code and infraspecifics without a code (NA). _2026-06-12_
3. Duplicate USDA codes. Many of these have been resolved by removing outdated taxa and LTER codes, or by finding appropriate synonym codes for infraspecific taxa. BOCO2 remains because there are not separate infraspecific taxa in USDA Plants for these taxa (and they are in Allred).
4. USDA code synonyms are marked (`usda_code_is_syn` column) and there are 28 synonym codes. These may indicate a taxon in the list that has had a taxonomic change that JRN doesn't recognize yet. This can be a simple naming change, or a "lumping" event (two species or infraspecifics merged into one). In the former case it may make sense to keep the current LTER code until the field crew is ready to accept the naming change. In the latter case, we may want to consider retiring the outdated taxon and its LTER code, especially if the outdated taxon is not commonly called in the field. Sometimes both a naming and lumping event have occurred, which is tricky. There are a few notes for these above, summarizeable as:
    - PORE & PHIN are out of date and can probably be retired (subsumed into POOL and PHCO, respectively)
    - TAAN and TAAU and corresponding USDA codes have merged and been renamed in USDA Plants (both in PHAU13) but not Allred
    - BOTO2 has been subsumed into BOSP in USDA Plants, but not Allred
    - TECOA has been subsumed into TECO, unclear in Allred
    - ERCUC2 and ERCUC4 have merged into ERCU2 in USDA Plants, but not in Allred
    - ECTRG2 (ECTR LTER code dupe) is subsumed into ECCOG in USDA Plants, but Allred disputes and assigns ECCOC
5. Orphaned or missing taxa: What happened to RHBE? It was in Plntalfa, but disappeared in later lists. No notes found. ...RESOLVED 2026-07-21 - was a mistake
6. The `lter_observed` column approximately matches Plntalfa and John's earlier list. Should we use the "present" subset for a website list?
    - still unclear _2026-06-12_
    - RESOLVED 2026-07-21 - there is a new sheet in the main list
7. For plant traits, does it make sense to apply any at the genus of family level?
8. There are a few other taxonomy issues to figure out. See the other notes section below.

## Other notes

* cpath column came from research from John Ludwig, with minor changes from John Anderson. The metadata for EDI dataset should say this. We should cite the publication with this information.
* Ferns from John’s list came from IBP surveys
* John Ludwig should be listed as an authors for starting original Plntalfa list (we can check to see if Gary Cunningham should be infolved)
* Change Acacia constricta var constricta to just species ...RESOLVED 2026-07-21
* Check MESC - do we need var laevis? Kelley doesn't have it so probably drop ...RESOLVED 2026-07-21
* Note that SOIL and ROAD are for lter transect plant line intercept, NONE means no plants

**Uncertain infraspecific taxa IDs**

* _Astragalus nutallianus var. austrinus_ (USDA code ASNUA, LTER code ASNU) was present in John's list, but Allred does not confirm the infraspecific.
    -  ...RESOLVED 2026-07-21 - leaving as is because it likeliest var based on Flora Neomexicana
* _Astragalus wootonii_ (USDA code ASWO2, LTER code ASWO) is called _Astragalus allochrous var. playanus_ (ASALP USDA code) by Allred.
    - change to Allred taxonomy, with ASALP LTER code, and move ASWO2 to synonyms list - note we use Allred as authority ...RESOLVED 2026-07-21
* _Menodora scabra var. scabra_ was listed as the infraspecific taxa in the EDI list but I can't find it anywhere (USDA Plants or Allred)
    -  ...RESOLVED 2026-07-21 ignore for now and only list to species
* For _Verbena wrightii_ (VEWR, or GLBIC in USDA Plants), note that Allred lists _Glandularia wrightii_ and _Glandularia pubera_ (GLBIB2) and, but _pubera_ seems to be subsumed into GLBIC according to USDA Plants. Also note VEAM (_V. ambrosifolia_) has been removed.
    - ...RESOLVED 2026-07-21 - VEAM will be removed from plntalfa and we don't use both species at JRN

**Data source notes**

Its worth noting that Allred very often is unaware of, or just not using, an up-to-date USDA code, especially for infraspecific rank taxa. There are also some typos in USDA codes and binomial names. Some of these are corrected in v4 of Greg's Allred table. 

Also note that **John's plant list** and the **EDI plant list** fairly often have infraspecific taxa that are not matched to the USDA code they use. Often these infraspecific taxa are not found in Allred. Example: LTER code ASTE.


In [30]:
system("jupyter nbconvert './plant_list_notes_and_val.ipynb' --no-input --to html")